### Stock RAG Agent Quant Kavin

In [11]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.tools import QueryEngineTool
from llama_index.core.agent.workflow import ReActAgent
from llama_index.llms.openai import OpenAI
import os

In [ ]:
os.environ["OPENAI_API_KEY"] = "insert you key"

In [13]:
#.\stock_agent_env\Scripts\Activate
#echo $env:OPENAI_API_KEY

In [ ]:
OPENAI_API_KEY = "insert your key"

In [ ]:
goog_file = [r"raw string of file"]

In [5]:
#embedding docs
#initialize OpenAi embedding model
#Turn into RAG agent
#Create tool to utlize previous code
#initialize openai model with system prompt
#create task prompt
#get output and print output

In [6]:
#PDF Embedding

In [7]:
doc_list = []

for doc in goog_file:
    docs = SimpleDirectoryReader(input_files =[doc]).load_data()
    doc_list.extend(docs)

In [8]:
#OpenAi Embedding model

In [16]:
openai_model = OpenAIEmbedding(model_name='text-embedding-3-small', api_key=OPENAI_API_KEY)
Settings.openai_model = openai_model

In [17]:
index = VectorStoreIndex.from_documents(doc_list)

2026-02-14 14:27:12,812 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-14 14:27:14,230 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [25]:
openai_llm = OpenAI(api_key=OPENAI_API_KEY,model='gpt-4.1-mini')

In [26]:
Settings.openai_llm = openai_llm

In [27]:
query_engine = index.as_query_engine()

In [28]:
query_tool = QueryEngineTool.from_defaults(query_engine=query_engine,
name="sec_filing_retriever",
description=("Use this tool to retrieve financial, risk, revenue, "
        "cash flow, and management discussion details from SEC filings.")) 

In [29]:
ai_agent = ReActAgent(
tools=[query_tool],
llm=openai_llm,
verbose=True,
system_prompt=""" 
You are a senior financial risk analyst specializing in SEC 10-K filings.

Your objective is to analyze the provided 10-K document and produce a structured, evidence-based risk assessment.

CRITICAL INSTRUCTIONS:

1. You MUST use the 'document_retrieval_tool' to access all document information. 
   Do NOT rely on prior knowledge or assumptions.
   If information is missing, explicitly state that it was not found in the document.

2. Focus on identifying and analyzing:
   - Business model risks
   - Revenue concentration risks
   - Competitive and industry risks
   - Regulatory and legal risks
   - Debt and liquidity risks
   - Cash flow instability
   - Operational risks
   - Macroeconomic exposure
   - Litigation or contingent liabilities
   - Management discussion of uncertainties

3. When summarizing risks:
   - Distinguish between structural risks (long-term) and cyclical risks (temporary).
   - Highlight recurring themes or repeated warnings.
   - Assess severity, likelihood, and financial impact.

4. Risk Scoring Methodology (1-5 scale):
   1 = Minimal risk, strong balance sheet, stable operations
   2 = Low risk, manageable exposures
   3 = Moderate risk, notable vulnerabilities
   4 = High risk, material financial or operational concerns
   5 = Severe risk, substantial solvency or structural threats

   Base the score on:
   - Magnitude of financial exposure
   - Debt burden and liquidity profile
   - Consistency of risk disclosures
   - Dependence on external conditions
   - Concentration risks
   - Management tone regarding uncertainty

5. Provide:
   - A structured summary of key risks
   - Analytical reasoning explaining how the risks justify the score
   - Clear references to themes identified in the document

6. Output Formatting Requirement:
   You MUST format the final score exactly as:

   Overall Risk Score:***[score goes here]***

   Replace [score goes here] with a single integer between 1 and 5.
   Do not add anything inside the asterisks except the number.

"""
)

In [35]:
response = await ai_agent.run(
    user_msg=""" 
Perform a comprehensive risk analysis of the 10-K filing for Alphabet Inc.

You MUST use the 'document_retrieval_tool' to retrieve all relevant information from the document. 
Do not rely on external knowledge.

Follow this structured workflow:

1. Identify the company name and filing date.
2. Retrieve and analyze:
   - Item 1A: Risk Factors
   - Management Discussion and Analysis (MD&A)
   - Liquidity and Capital Resources
   - Debt obligations and maturity structure
   - Legal proceedings
   - Any disclosed uncertainties or forward-looking risk statements

3. Categorize risks into:
   - Financial risks
   - Operational risks
   - Regulatory/legal risks
   - Competitive/industry risks
   - Macroeconomic risks
   - Concentration risks

4. For each category:
   - Summarize the key issues
   - Assess severity (low/moderate/high)
   - Note whether risks appear recurring or structural

5. Based strictly on retrieved evidence, assign an overall risk score (1–5).
   Justify the score using:
   - Debt levels and liquidity profile
   - Revenue stability or concentration
   - Frequency and tone of risk disclosures
   - Exposure to external market conditions

Conclude with:

Overall Risk Score:***[single integer 1-5]***
"""
)
print(response)

2026-02-14 15:20:23,813 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-14 15:20:26,082 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-14 15:20:27,429 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-14 15:20:27,748 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-14 15:20:29,906 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-14 15:20:30,397 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-14 15:20:30,660 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-14 15:20:33,012 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-14 15:20:34,216 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2

Overall Risk Score:***3***
